# Obstacle Pipeline — SAM3 + DA3, Step-by-step Visualizations

Self-contained walk-through of every pipeline stage.  
No imports from `obstacle_detection.*` — all logic is inlined so each
cell is independently readable and the visualizations make every
decision visible.

| Stage | What happens |
|---|---|
| 1 | Load image |
| 2 | SAM3 text-prompted segmentation |
| 3 | Vehicle (car) detection |
| 4 | Depth-Anything-3 depth estimation |
| 5 | Identify the car SAM mask |
| 6 | Rule filters — min-area, car-part, foreground, ground, touch |
| 7 | Occlusion depth-outlier mechanism |
| 8 | Final result — 5-panel comparison |
| 9 | Rule-filter summary table |

**Depth convention:** `depth_map` is metric depth → **smaller = closer**.  
A foreground object therefore has a depth **smaller** than the car's average depth.

## 0 · Config & paths

In [ ]:
import os, sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

# ── server paths (Azure GPU server) ──────────────────────────────────────────
BASE_PATH       = "/mnt/aitraining/krishna/2026/obstacle_pipeline"
TEST_IMAGES_DIR = os.path.join(BASE_PATH, "test_images")
OBSTACLE_DIR    = os.path.join(TEST_IMAGES_DIR, "obstacle")
SAM3_WEIGHTS    = "sam3.pt"
DA3_REPO        = os.path.join(BASE_PATH, "Depth-Anything-3")
DA3_MODEL_NAME  = "depth-anything/DA3-LARGE-1.1"
DA3_CACHE_DIR   = os.path.join(BASE_PATH, "da3_model")

# ── obstacle-decision thresholds (mirrors ObstacleConfig defaults) ────────────
CFG = dict(
    enable_min_area_filter   = True,
    enable_car_part_filter   = True,
    enable_foreground_filter = True,
    enable_ground_filter     = True,
    enable_touch_filter      = True,
    use_occlusion_depth      = True,
    min_area_ratio           = 0.0015,
    car_part_containment     = 0.5,
    depth_margin             = 0.05,
    ground_grad_ratio        = 0.15,
    ground_bottom_frac       = 0.85,
    ground_min_bottom_width  = 0.70,
    occlusion_depth_margin   = 0.08,
    occlusion_min_area_ratio = 0.003,
    occlusion_band_ratio     = 0.02,
    occlusion_erode_px       = 0,
    sam3_text_prompts = [
        "person", "child", "bicycle", "motorcycle", "stroller", "wheelchair",
        "shopping cart", "trolley", "dog", "cat", "animal", "traffic cone",
        "pole", "bollard", "trash can", "box", "bag", "ladder", "chair",
        "plant", "bush", "ball", "toy",
    ],
    sam3_conf = 0.25,
)

# ── pick a test image ─────────────────────────────────────────────────────────
images = sorted(f for f in os.listdir(OBSTACLE_DIR) if f.endswith((".png", ".jpg")))
print(f"Obstacle images available: {len(images)}")
for i, n in enumerate(images):
    print(f"  [{i}] {n}")

IMAGE_INDEX = 0       # ← change to try different images
IMAGE_PATH  = os.path.join(OBSTACLE_DIR, images[IMAGE_INDEX])
print(f"\nSelected: {IMAGE_PATH}")

## 1 · Load image

In [ ]:
original_bgr = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
original_img = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
oh, ow = original_img.shape[:2]
print(f"Image size: {ow} × {oh}  |  {os.path.basename(IMAGE_PATH)}")

plt.figure(figsize=(10, 6))
plt.imshow(original_img)
plt.title(f"Step 1 — Original image\n{os.path.basename(IMAGE_PATH)}", fontsize=12)
plt.axis("off")
plt.tight_layout()
plt.show()

## 2 · SAM3 text-prompted segmentation

SAM3 has **no "segment everything" mode** — it only responds to explicit text
prompts. We feed it the obstacle concept vocabulary and it returns one mask per
detected instance of those named classes.

In [ ]:
from ultralytics.models.sam import SAM3SemanticPredictor

sam3_overrides = dict(
    model   = SAM3_WEIGHTS,
    conf    = CFG["sam3_conf"],
    task    = "segment",
    mode    = "predict",
    half    = True,
    verbose = False,
    save    = False,   # don't write annotated images to runs/
    imgsz   = 644,     # nearest multiple of SAM3's stride-14 above 640
)
sam3_predictor = SAM3SemanticPredictor(overrides=sam3_overrides)
sam3_predictor.set_image(IMAGE_PATH)
sam3_results_list = sam3_predictor(text=CFG["sam3_text_prompts"])
sam_res = sam3_results_list[0] if isinstance(sam3_results_list, (list, tuple)) else sam3_results_list

sam_masks_raw = sam_res.masks.data.cpu().numpy() if sam_res.masks is not None else np.array([])
n_masks = len(sam_masks_raw)
print(f"SAM3 returned {n_masks} mask(s) for {len(CFG['sam3_text_prompts'])} concept prompts")

def _name_for(names, idx):
    """Resolve a class name whether `names` is a dict {id: name} or a list [name, ...]."""
    if isinstance(names, dict):
        return names.get(idx, str(idx))
    if isinstance(names, (list, tuple)) and 0 <= idx < len(names):
        return names[idx]
    return str(idx)

if sam_res.boxes is not None and sam_res.names:
    cls_ids   = sam_res.boxes.cls.cpu().numpy().astype(int)
    cls_names = [_name_for(sam_res.names, c) for c in cls_ids]
    confs     = sam_res.boxes.conf.cpu().numpy()
    print("\nDetected concepts:")
    for i, (nm, cf) in enumerate(zip(cls_names, confs)):
        print(f"  mask {i:>2}: {nm:<20} conf={cf:.2f}")
else:
    cls_names = [f"obj{i}" for i in range(n_masks)]
    confs     = np.ones(n_masks)

In [ ]:
# ── ultralytics annotated plot ────────────────────────────────────────────────
sam_img_bgr  = sam_res.plot()
sam_img_rgb  = cv2.cvtColor(sam_img_bgr, cv2.COLOR_BGR2RGB)
sam_img_disp = cv2.resize(sam_img_rgb, (ow, oh), interpolation=cv2.INTER_LINEAR)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].imshow(original_img);  axes[0].set_title("Original",           fontsize=11); axes[0].axis("off")
axes[1].imshow(sam_img_disp);  axes[1].set_title("SAM3 segmentation",  fontsize=11); axes[1].axis("off")
plt.suptitle("Step 2 — SAM3 text-prompted segmentation", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ── per-mask grid ─────────────────────────────────────────────────────────────
def show_mask_grid(img_rgb, masks, titles=None, max_cols=4, color=(255, 80, 0),
                   alpha=0.55, figsize_per=(4.5, 3.2)):
    n = len(masks)
    if n == 0:
        print("No masks returned."); return
    cols  = min(n, max_cols)
    rows  = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols,
                             figsize=(figsize_per[0]*cols, figsize_per[1]*rows))
    axes = np.array(axes).flatten()
    h2, w2 = img_rgb.shape[:2]
    c = np.array(color, dtype=float)
    for idx, m in enumerate(masks):
        mr  = cv2.resize(m.astype(np.uint8), (w2, h2), interpolation=cv2.INTER_NEAREST).astype(bool)
        ov  = img_rgb.copy()
        ov[mr] = (ov[mr] * (1 - alpha) + c * alpha).astype(np.uint8)
        axes[idx].imshow(ov)
        axes[idx].set_title(titles[idx] if titles else f"mask {idx}", fontsize=8)
        axes[idx].axis("off")
    for ax in axes[n:]: ax.set_visible(False)
    plt.suptitle(f"SAM3 — {n} individual mask(s)", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

mask_titles = [f"#{i} {cls_names[i]}\nconf={confs[i]:.2f}" for i in range(n_masks)]
show_mask_grid(original_img, sam_masks_raw, titles=mask_titles, max_cols=4)

## 3 · Vehicle (car) detection

In [ ]:
from obstacle_detection.handlers import VehicleModelHandler

veh_handler = VehicleModelHandler()
veh_res     = veh_handler.infer(IMAGE_PATH)

n_veh = len(veh_res.instances) if hasattr(veh_res, "instances") else 0
print(f"Vehicle detections: {n_veh}")
if n_veh > 0 and len(veh_res.bboxes) > 0:
    print(f"  bbox (x1,y1,x2,y2): {veh_res.bboxes[0]}")

veh_img_disp = cv2.resize(veh_res.draw(original_img.copy()), (ow, oh),
                          interpolation=cv2.INTER_LINEAR)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].imshow(original_img);  axes[0].set_title("Original",          fontsize=11); axes[0].axis("off")
axes[1].imshow(veh_img_disp);  axes[1].set_title("Vehicle detection", fontsize=11); axes[1].axis("off")
plt.suptitle("Step 3 — Vehicle (car) detection", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 4 · Depth-Anything-3 (DA3) depth estimation

In [ ]:
os.environ["HF_HOME"] = os.environ.get("HF_HOME", DA3_CACHE_DIR)
if DA3_REPO not in sys.path:
    sys.path.append(DA3_REPO)

from depth_anything_3.api            import DepthAnything3
from depth_anything_3.utils.visualize import visualize_depth

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

da3_model = DepthAnything3.from_pretrained(DA3_MODEL_NAME).to(device=device)
da3_res   = da3_model.inference([original_img])

depth_map  = da3_res.depth[0] if len(da3_res.depth.shape) == 3 else da3_res.depth
img_h, img_w = depth_map.shape
da3_colored  = visualize_depth(depth_map)
da3_disp     = cv2.resize(da3_colored, (ow, oh), interpolation=cv2.INTER_LINEAR)
print(f"Depth map: {depth_map.shape}  min={depth_map.min():.3f}  max={depth_map.max():.3f} m")

fig, axes = plt.subplots(1, 3, figsize=(24, 6))
axes[0].imshow(original_img)
axes[0].set_title("Original", fontsize=11); axes[0].axis("off")

axes[1].imshow(da3_disp)
axes[1].set_title("Depth map (DA3)\nbrighter = closer", fontsize=11); axes[1].axis("off")

im = axes[2].imshow(depth_map, cmap="plasma")
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
axes[2].set_title("Raw metric depth (m)\nsmaller = closer", fontsize=11); axes[2].axis("off")

plt.suptitle("Step 4 — Depth-Anything-3 depth estimation", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 5 · Identify the car SAM mask

We select the SAM3 mask with the **highest IoU against the vehicle detection region**
(segmentation mask preferred over bbox). When the vehicle model did not fire we fall
back to a SAM-only heuristic: the largest near object (area / median_depth).

The chosen car mask is then **clipped to the vehicle silhouette** to remove any
ground/shadow pixels SAM merged into the car blob.

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
CAR_COLOR = np.array([0, 200, 255])   # cyan

def vehicle_region_bin(veh_result, h, w):
    n = len(veh_result.instances) if hasattr(veh_result, "instances") else 0
    if n == 0: return None
    if len(veh_result.masks) > 0:
        vm = veh_result.masks[0]
        if vm.shape != (h, w):
            vm = cv2.resize(vm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        return vm > 0.5
    if len(veh_result.bboxes) > 0:
        x1, y1, x2, y2 = map(int, veh_result.bboxes[0])
        vb = np.zeros((h, w), dtype=bool)
        vb[y1:y2, x1:x2] = True
        return vb
    return None

def is_ground(mask_bin, depth, thresh=0.15):
    ys, xs = np.where(mask_bin)
    if len(ys) == 0: return True
    mid_y = (ys.min() + ys.max()) / 2.0
    top, bot = ys < mid_y, ys >= mid_y
    if not (top.any() and bot.any()): return False
    own_d = np.median(depth[mask_bin]) + 1e-6
    return (np.median(depth[ys[top], xs[top]]) - np.median(depth[ys[bot], xs[bot]])) / own_d > thresh

def overlay_mask(img, mask_hw, color, alpha=0.5):
    out = img.copy()
    mr  = cv2.resize(mask_hw.astype(np.uint8), (img.shape[1], img.shape[0]),
                     interpolation=cv2.INTER_NEAREST).astype(bool)
    out[mr] = (out[mr] * (1 - alpha) + np.array(color) * alpha).astype(np.uint8)
    return out

# ── resize SAM masks to depth-map resolution ─────────────────────────────────
sam_masks = np.array([
    cv2.resize(m.astype(np.uint8), (img_w, img_h), interpolation=cv2.INTER_NEAREST)
    for m in sam_masks_raw
]) if n_masks > 0 else np.array([])

# ── vehicle region ────────────────────────────────────────────────────────────
v_bin = vehicle_region_bin(veh_res, img_h, img_w)

# ── pick car mask by max IoU with vehicle region ──────────────────────────────
best_iou, car_idx, ious = 0.0, -1, []
if v_bin is not None and v_bin.any() and n_masks > 0:
    for i, s in enumerate(sam_masks):
        sb    = s > 0.5
        inter = np.logical_and(v_bin, sb).sum()
        union = np.logical_or(v_bin,  sb).sum()
        iou   = inter / union if union > 0 else 0.0
        ious.append(iou)
        if iou > best_iou:
            best_iou, car_idx = iou, i
    seed_src = "vehicle" if car_idx != -1 else "SAM-only"
else:
    ious = [0.0] * n_masks
    seed_src = "SAM-only"

# ── SAM-only fallback ─────────────────────────────────────────────────────────
if car_idx == -1 and n_masks > 0:
    img_area = img_h * img_w
    best_s = -1.0
    for i, m in enumerate(sam_masks):
        b = m > 0.5
        if b.sum() < img_area * 0.02 or is_ground(b, depth_map):
            continue
        score = b.sum() / (np.median(depth_map[b]) + 1e-6)
        if score > best_s:
            best_s, car_idx = score, i

assert car_idx != -1, "No car mask found — check the image or lower thresholds"

car_mask_bin_raw = sam_masks[car_idx] > 0.5
if v_bin is not None and v_bin.any():
    clipped = np.logical_and(car_mask_bin_raw, v_bin)
    car_mask_bin = clipped if clipped.sum() > 0 else car_mask_bin_raw
else:
    car_mask_bin = car_mask_bin_raw

car_depth = float(np.mean(depth_map[car_mask_bin]))
print(f"Car: SAM mask #{car_idx}  IoU={best_iou:.3f}  seed={seed_src}  avg_depth={car_depth:.3f} m")

In [ ]:
# ── IoU bar chart + overlays ──────────────────────────────────────────────────
VEHICLE_C = np.array([0, 255, 120])   # green

car_ov = overlay_mask(original_img, car_mask_bin, CAR_COLOR)
if v_bin is not None:
    car_ov = overlay_mask(car_ov, v_bin, VEHICLE_C, alpha=0.25)

# raw vs clipped side-by-side
raw_ov = overlay_mask(original_img, car_mask_bin_raw, [200, 200, 0], alpha=0.5)  # yellow = raw

fig, axes = plt.subplots(1, 3, figsize=(24, 6))

axes[0].imshow(car_ov)
patches = [mpatches.Patch(color=CAR_COLOR/255, label=f"Car mask #{car_idx} (clipped, IoU={best_iou:.3f})"),
           mpatches.Patch(color=VEHICLE_C/255, label="Vehicle detection region")]
axes[0].legend(handles=patches, loc="upper right", fontsize=8)
axes[0].set_title(f"Car mask — seed={seed_src}", fontsize=11); axes[0].axis("off")

if n_masks > 0 and any(v > 0 for v in ious):
    bar_colors = ["crimson" if i == car_idx else "steelblue" for i in range(n_masks)]
    axes[1].bar([f"m{i}\n{cls_names[i] if i < len(cls_names) else ''}" for i in range(n_masks)],
                ious, color=bar_colors)
    axes[1].set_ylabel("IoU vs vehicle region")
    axes[1].set_title(f"IoU per SAM mask vs vehicle region\n(red = selected car mask #{car_idx})", fontsize=11)
    axes[1].tick_params(axis='x', labelsize=7)
else:
    axes[1].text(0.5, 0.5, "No IoU data\n(SAM-only fallback)", ha="center", va="center")
    axes[1].axis("off")

im = axes[2].imshow(depth_map, cmap="plasma")
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
axes[2].set_title(f"Metric depth | car avg = {car_depth:.3f} m", fontsize=11); axes[2].axis("off")

plt.suptitle("Step 5 — Car mask identification", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 6 · Obstacle rule filters — step-by-step

Each rule is applied in sequence; masks that fail are coloured **grey** in the
per-rule comparison. Survivors advance in **orange**.

| # | Rule | What it drops |
|---|------|---------------|
| 1 | `min_area` | specks smaller than `min_area_ratio` of image |
| 2 | `car_part` | masks mostly inside the vehicle silhouette |
| 3 | `foreground` | masks at or behind the car depth |
| 4 | `ground` | receding floor plane |
| 5 | `touch` | masks not touching the car mask |

In [ ]:
# ── shared context ────────────────────────────────────────────────────────────
img_area         = img_h * img_w
foreground_thresh = car_depth * (1.0 - CFG["depth_margin"])
car_part_ref     = v_bin if v_bin is not None else car_mask_bin
car_touch_ref    = cv2.dilate(car_mask_bin.astype(np.uint8),
                               np.ones((3, 3), np.uint8), iterations=1).astype(bool)

candidates = [i for i in range(len(sam_masks)) if i != car_idx]
surviving  = set(candidates)
rule_log   = {}
print(f"car_depth={car_depth:.3f}  foreground_thresh={foreground_thresh:.3f}")
print(f"Candidate masks (excl. car #{car_idx}): {candidates}")

# ── comparison-plot helper ────────────────────────────────────────────────────
KEEP_C = np.array([255, 120,  30])   # orange — pass
FAIL_C = np.array([120, 120, 120])   # grey   — reject

def show_rule(rule_name, before, decisions, img, masks, extra_ax_fn=None):
    def build(verdict_map):
        h2, w2 = img.shape[:2]
        out = img.copy()
        cr  = cv2.resize(car_mask_bin.astype(np.uint8), (w2, h2),
                         interpolation=cv2.INTER_NEAREST).astype(bool)
        out[cr] = (out[cr] * 0.5 + CAR_COLOR * 0.5).astype(np.uint8)
        for i in before:
            mr = cv2.resize(masks[i].astype(np.uint8), (w2, h2),
                            interpolation=cv2.INTER_NEAREST).astype(bool)
            c  = KEEP_C if verdict_map.get(i, True) else FAIL_C
            out[mr] = (out[mr] * 0.45 + c * 0.55).astype(np.uint8)
        return out
    pass_n = sum(1 for v in decisions.values() if v)
    ncols  = 3 if extra_ax_fn else 2
    fig, axes = plt.subplots(1, ncols, figsize=(8 * ncols, 5))
    axes[0].imshow(build({i: True for i in before}))
    axes[0].set_title(f"Before [{rule_name}]\n{len(before)} candidate(s)", fontsize=11)
    axes[0].axis("off")
    axes[1].imshow(build(decisions))
    axes[1].set_title(f"After [{rule_name}]\n{pass_n} pass (orange)   {len(before)-pass_n} reject (grey)",
                      fontsize=11)
    axes[1].axis("off")
    patches = [mpatches.Patch(color=CAR_COLOR/255, label="Car"),
               mpatches.Patch(color=KEEP_C/255,    label="Pass"),
               mpatches.Patch(color=FAIL_C/255,    label="Reject")]
    axes[0].legend(handles=patches, loc="upper right", fontsize=8)
    if extra_ax_fn: extra_ax_fn(axes[2])
    plt.suptitle(f"Rule: {rule_name}", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

In [ ]:
# ── Rule 1: min_area ──────────────────────────────────────────────────────────
before = list(surviving); decisions = {}
for i in before:
    sb   = sam_masks[i] > 0.5
    ar   = int(sb.sum())
    rat  = ar / img_area
    ok   = (rat >= CFG["min_area_ratio"]) if CFG["enable_min_area_filter"] else True
    decisions[i] = ok
    print(f"  mask {i}: {'PASS' if ok else 'REJECT':6}  area_ratio={rat:.4f} (min={CFG['min_area_ratio']})")

surviving = {i for i, p in decisions.items() if p}
rule_log["min_area"] = decisions
show_rule("min_area", before, decisions, original_img, sam_masks)

In [ ]:
# ── Rule 2: car_part ─────────────────────────────────────────────────────────
before = list(surviving); decisions = {}
for i in before:
    sb  = sam_masks[i] > 0.5
    ar  = int(sb.sum())
    cnt = np.logical_and(sb, car_part_ref).sum() / ar if car_part_ref is not None else 0.0
    ok  = (cnt <= CFG["car_part_containment"]) if CFG["enable_car_part_filter"] else True
    decisions[i] = ok
    print(f"  mask {i}: {'PASS' if ok else 'REJECT':6}  containment={cnt:.2%} (max={CFG['car_part_containment']:.0%})")

surviving = {i for i, p in decisions.items() if p}
rule_log["car_part"] = decisions
show_rule("car_part", before, decisions, original_img, sam_masks)

In [ ]:
# ── Rule 3: foreground (depth) ────────────────────────────────────────────────
before = list(surviving); decisions = {}; mask_depths = {}
for i in before:
    sb = sam_masks[i] > 0.5
    md = float(np.mean(depth_map[sb]))
    mask_depths[i] = md
    ok = (md < foreground_thresh) if CFG["enable_foreground_filter"] else True
    decisions[i] = ok
    kind = "FG" if md < car_depth else ("~car" if md < foreground_thresh else "BG")
    print(f"  mask {i}: {'PASS' if ok else 'REJECT':6}  depth={md:.3f} thresh={foreground_thresh:.3f} car={car_depth:.3f} [{kind}]")

surviving = {i for i, p in decisions.items() if p}
rule_log["foreground"] = decisions

def depth_bar_ax(ax):
    idxs = list(mask_depths.keys())
    vals = [mask_depths[i] for i in idxs]
    cols = ["limegreen" if decisions[i] else "tomato" for i in idxs]
    ax.bar([f"m{i}" for i in idxs], vals, color=cols)
    ax.axhline(car_depth,         color="cyan",   ls="--", lw=2, label=f"car_depth={car_depth:.3f} m")
    ax.axhline(foreground_thresh, color="orange", ls=":",  lw=2, label=f"fg_thresh={foreground_thresh:.3f} m")
    ax.set_ylabel("mean depth (m)"); ax.set_title("Depth per mask (green=pass)")
    ax.legend(fontsize=8)

show_rule("foreground", before, decisions, original_img, sam_masks, extra_ax_fn=depth_bar_ax)

In [ ]:
# ── Rule 4: ground plane ──────────────────────────────────────────────────────
before = list(surviving); decisions = {}; ground_details = {}
for i in before:
    sb  = sam_masks[i] > 0.5
    md  = float(np.mean(depth_map[sb]))
    ys, xs = np.where(sb)
    bot_band  = ys >= CFG["ground_bottom_frac"] * img_h
    bot_width = (np.unique(xs[bot_band]).size / img_w) if bot_band.any() else 0.0
    delta = 0.0; is_gnd = False
    if bot_width >= CFG["ground_min_bottom_width"]:
        mid_y = (ys.min() + ys.max()) / 2.0
        top, bot = ys < mid_y, ys >= mid_y
        if top.any() and bot.any():
            top_d = float(np.median(depth_map[ys[top], xs[top]]))
            bot_d = float(np.median(depth_map[ys[bot], xs[bot]]))
            delta = (top_d - bot_d) / (md + 1e-6)
            is_gnd = delta > CFG["ground_grad_ratio"]
    ok  = (not is_gnd) if CFG["enable_ground_filter"] else True
    decisions[i] = ok
    ground_details[i] = (bot_width, delta)
    print(f"  mask {i}: {'PASS' if ok else 'REJECT':6}  bot_width={bot_width:.2f}  delta_ratio={delta:.3f}")

surviving = {i for i, p in decisions.items() if p}
rule_log["ground"] = decisions

def ground_bar_ax(ax):
    idxs   = list(ground_details.keys())
    deltas = [ground_details[i][1] for i in idxs]
    cols   = ["limegreen" if decisions[i] else "tomato" for i in idxs]
    ax.bar([f"m{i}" for i in idxs], deltas, color=cols)
    ax.axhline(CFG["ground_grad_ratio"], color="orange", ls="--", lw=2,
               label=f"ground_grad_ratio={CFG['ground_grad_ratio']}")
    ax.set_ylabel("vertical depth delta ratio")
    ax.set_title("Ground filter: delta ratio per mask\n(above threshold + wide bottom = ground)")
    ax.legend(fontsize=8)

show_rule("ground", before, decisions, original_img, sam_masks, extra_ax_fn=ground_bar_ax)

In [ ]:
# ── Rule 5: touch ─────────────────────────────────────────────────────────────
TOUCH_C = np.array([255, 255, 0])   # yellow
before  = list(surviving); decisions = {}
for i in before:
    sb    = sam_masks[i] > 0.5
    touch = np.logical_and(sb, car_touch_ref).any()
    ok    = touch if CFG["enable_touch_filter"] else True
    decisions[i] = ok
    print(f"  mask {i}: {'PASS' if ok else 'REJECT':6}  touches_car={touch}")

surviving = {i for i, p in decisions.items() if p}
rule_log["touch"] = decisions

# touch-zone visualisation
touch_ov = original_img.copy()
tr = cv2.resize(car_touch_ref.astype(np.uint8), (ow, oh), interpolation=cv2.INTER_NEAREST).astype(bool)
cr = cv2.resize(car_mask_bin.astype(np.uint8),  (ow, oh), interpolation=cv2.INTER_NEAREST).astype(bool)
touch_ov[tr] = (touch_ov[tr] * 0.5 + TOUCH_C   * 0.5).astype(np.uint8)
touch_ov[cr] = (touch_ov[cr] * 0.5 + CAR_COLOR  * 0.5).astype(np.uint8)

def touch_zone_ax(ax):
    ax.imshow(touch_ov)
    ax.set_title("Car mask (cyan) + touch zone (yellow, 1px dilation)")
    ax.axis("off")
    ax.legend(handles=[
        mpatches.Patch(color=CAR_COLOR/255, label="Car mask"),
        mpatches.Patch(color=TOUCH_C/255,   label="Touch zone (1px dilated)"),
    ], loc="upper right", fontsize=8)

show_rule("touch", before, decisions, original_img, sam_masks, extra_ax_fn=touch_zone_ax)

obstacle_mask_indices = list(surviving)
mask_obstacle_exist   = len(obstacle_mask_indices) > 0
print(f"\nMask-overlap result: obstacle_exist={mask_obstacle_exist}  indices={obstacle_mask_indices}")

## 7 · Occlusion depth-outlier mechanism

Independent of SAM mask quality. Scans the **holes inside the car's convex hull**
for pixels whose depth is anomalously **closer** than the plane-detrended car surface.
A genuine occluder hides car pixels behind it, so those holes appear closer than expected.

Intermediate steps visualised: hull → plane fit → outliers → qualifying band.

In [ ]:
occ_exist, occlusion_mask_bin = False, np.zeros((img_h, img_w), dtype=bool)

if CFG["use_occlusion_depth"]:
    ys_c, xs_c = np.where(car_mask_bin)

    # 1. convex hull of car mask
    pts  = np.column_stack([xs_c, ys_c]).astype(np.int32)
    hull = cv2.convexHull(pts)
    hull_region = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillConvexPoly(hull_region, hull, 1)
    hull_bin = hull_region.astype(bool)

    # 2. plane-fit expected car depth
    A = np.column_stack([xs_c, ys_c, np.ones(len(xs_c))]).astype(np.float64)
    z = depth_map[car_mask_bin].astype(np.float64)
    (a, b_p, c_p), *_ = np.linalg.lstsq(A, z, rcond=None)
    yy, xx   = np.mgrid[0:img_h, 0:img_w]
    expected = a * xx + b_p * yy + c_p
    closer_amt = expected - depth_map    # >0 means CLOSER than fitted plane

    # 3. holes = hull pixels NOT on genuine car surface
    car_region = car_mask_bin | (v_bin if v_bin is not None else np.zeros_like(car_mask_bin))
    holes   = hull_bin & ~car_region
    margin  = car_depth * CFG["occlusion_depth_margin"]
    outlier = holes & (closer_amt > margin)

    # 4. restrict to band around car + size gate
    band_px  = max(3, int(round(min(img_h, img_w) * CFG["occlusion_band_ratio"])))
    car_band = cv2.dilate(car_mask_bin.astype(np.uint8),
                          np.ones((band_px * 2 + 1,) * 2, np.uint8)).astype(bool)
    qualifying = outlier & car_band
    if CFG["occlusion_erode_px"] > 0 and qualifying.any():
        k = np.ones((CFG["occlusion_erode_px"]*2+1,)*2, np.uint8)
        qualifying = cv2.morphologyEx(qualifying.astype(np.uint8), cv2.MORPH_OPEN, k).astype(bool)

    total    = int(qualifying.sum())
    min_area = img_h * img_w * CFG["occlusion_min_area_ratio"]
    occ_exist = qualifying.any() and total >= min_area
    if occ_exist:
        occlusion_mask_bin = qualifying
        print(f"[RESULT] Occlusion obstacle — {total} px ({total/(img_h*img_w):.4f} of image), "
              f"median closer_amt={np.median(closer_amt[qualifying]):.3f} m")
    else:
        print(f"[INFO] Occlusion — no significant outliers ({total} px < {min_area:.0f})")

    # ── 4-step visual ─────────────────────────────────────────────────────────
    HULL_C = np.array([255, 200,  0])
    OUTL_C = np.array([255,   0,  0])
    QUAL_C = np.array([255,   0, 255])

    def tint(img, mask, color, alpha=0.55):
        out = img.copy()
        mr  = cv2.resize(mask.astype(np.uint8), (img.shape[1], img.shape[0]),
                         interpolation=cv2.INTER_NEAREST).astype(bool)
        out[mr] = (out[mr] * (1 - alpha) + np.array(color) * alpha).astype(np.uint8)
        return out

    p1 = tint(tint(original_img, car_mask_bin, CAR_COLOR, 0.5), hull_bin, HULL_C, 0.3)
    exp_vis = (expected - expected.min()) / (expected.ptp() + 1e-6)
    p2 = (plt.cm.plasma(exp_vis)[:, :, :3] * 255).astype(np.uint8)
    p2 = cv2.resize(p2, (ow, oh))
    p3 = tint(tint(original_img, car_mask_bin, CAR_COLOR, 0.5), outlier, OUTL_C, 0.7)
    p4 = tint(tint(original_img, car_mask_bin, CAR_COLOR, 0.5), qualifying, QUAL_C, 0.7)

    fig, axes = plt.subplots(1, 4, figsize=(30, 6))
    axes[0].imshow(p1)
    axes[0].set_title("1. Car (cyan) + Convex hull (yellow)", fontsize=10); axes[0].axis("off")
    axes[1].imshow(p2)
    axes[1].set_title("2. Plane-fitted expected car depth", fontsize=10); axes[1].axis("off")
    axes[2].imshow(p3)
    axes[2].set_title(f"3. Depth outliers in hull holes\n(red, closer_amt > {margin:.3f} m)", fontsize=10)
    axes[2].axis("off")
    axes[3].imshow(p4)
    result_str = f"OBSTACLE ({total} px)" if occ_exist else f"none ({total} px < {min_area:.0f})"
    axes[3].set_title(f"4. Qualifying pixels (magenta)\n{result_str}",
                      fontsize=10, color="red" if occ_exist else "green")
    axes[3].axis("off")
    plt.suptitle("Step 7 — Occlusion depth-outlier mechanism", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("[INFO] use_occlusion_depth=False — mechanism skipped")

## 8 · Final result — 5-panel comparison

In [ ]:
obstacle_exist  = mask_obstacle_exist or occ_exist
OBSTACLE_C      = np.array([255,   0,   0])
OCCLUSION_C     = np.array([255,   0, 255])

print(f"FINAL: obstacle_exist={obstacle_exist}  "
      f"(mask-overlap={mask_obstacle_exist}, depth-occlusion={occ_exist})")

# ── obstacle overlay ──────────────────────────────────────────────────────────
obs_ov = original_img.copy()
cr = cv2.resize(car_mask_bin.astype(np.uint8), (ow, oh), interpolation=cv2.INTER_NEAREST).astype(bool)
obs_ov[cr] = (obs_ov[cr] * 0.5 + CAR_COLOR * 0.5).astype(np.uint8)
for idx in obstacle_mask_indices:
    mr = cv2.resize(sam_masks[idx].astype(np.uint8), (ow, oh), interpolation=cv2.INTER_NEAREST).astype(bool)
    obs_ov[mr] = (obs_ov[mr] * 0.4 + OBSTACLE_C * 0.6).astype(np.uint8)
if occlusion_mask_bin.any():
    or_ = cv2.resize(occlusion_mask_bin.astype(np.uint8), (ow, oh), interpolation=cv2.INTER_NEAREST).astype(bool)
    obs_ov[or_] = (obs_ov[or_] * 0.4 + OCCLUSION_C * 0.6).astype(np.uint8)

# ── 5-panel figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(37, 7))
axes[0].imshow(original_img);  axes[0].set_title("Original",           fontsize=11); axes[0].axis("off")
axes[1].imshow(sam_img_disp);  axes[1].set_title("SAM3 segmentation",  fontsize=11); axes[1].axis("off")
axes[2].imshow(veh_img_disp);  axes[2].set_title("Vehicle detection",  fontsize=11); axes[2].axis("off")
axes[3].imshow(da3_disp);      axes[3].set_title("Depth map (DA3)",    fontsize=11); axes[3].axis("off")

axes[4].imshow(obs_ov)
if obstacle_exist:
    parts = []
    if obstacle_mask_indices:
        parts.append("masks " + ", ".join(f"#{i}" for i in obstacle_mask_indices))
    if occ_exist:
        parts.append("depth-occlusion (magenta)")
    axes[4].set_title("Car (cyan) + Obstacle:\n" + " | ".join(parts),
                      fontsize=10, color="red", fontweight="bold")
else:
    axes[4].set_title("Car (cyan) — No obstacle", fontsize=11, color="green", fontweight="bold")

legend_handles = [
    mpatches.Patch(color=CAR_COLOR/255,   label="Car mask (SAM3)"),
    mpatches.Patch(color=OBSTACLE_C/255,  label="Obstacle (mask-overlap)"),
    mpatches.Patch(color=OCCLUSION_C/255, label="Occluder (depth-outlier)"),
]
axes[4].legend(handles=legend_handles, loc="upper right", fontsize=8)

fig.suptitle(
    f"OBSTACLE EXIST: {obstacle_exist}   "
    f"(mask-overlap={mask_obstacle_exist}  |  depth-occlusion={occ_exist})",
    fontsize=16, fontweight="bold",
    color="red" if obstacle_exist else "green"
)
plt.tight_layout(); plt.show()

## 9 · Rule-filter summary table

In [ ]:
rules_order = ["min_area", "car_part", "foreground", "ground", "touch"]
all_cands   = [i for i in range(len(sam_masks)) if i != car_idx]

if not all_cands:
    print("No candidate masks to summarise.")
else:
    W = 11
    hdr = f"{'mask':>6}" + "".join(f"{r:>{W}}" for r in rules_order) + f"{'FINAL':>{W}}"
    print(hdr); print("-" * len(hdr))
    for i in all_cands:
        row = f"{i:>6}"
        for r in rules_order:
            if r not in rule_log:
                cell = "skipped"
            elif i not in rule_log[r]:
                cell = "(dropped)"
            elif rule_log[r][i]:
                cell = "PASS"
            else:
                cell = "REJECT"
            row += f"{cell:>{W}}"
        final = "OBSTACLE" if i in obstacle_mask_indices else "no"
        row += f"{final:>{W}}"
        print(row)

print(f"\nObstacle masks: {obstacle_mask_indices}")
print(f"Occlusion:      {occ_exist}")
print(f"Final verdict:  obstacle_exist={obstacle_exist}")